In [7]:
import os
import torch
import torchaudio
import numpy as np
import pickle
import ast
import math
import random
import logging
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

In [8]:
# ----------------------------------------------------
# 1. SETUP PATHS FOR KAGGLE DATASETS
# ----------------------------------------------------
DATASET_ROOT = "/kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0"
CHECKPOINT_PATH = "/kaggle/input/datasets/aatmajsandeeprakshe/f0-predictor-state/f0_predictor_state.pth"

train_datasets = {"ESD": os.path.join(DATASET_ROOT, "train")}
val_datasets = {"ESD": os.path.join(DATASET_ROOT, "val")}
test_datasets = {"ESD": os.path.join(DATASET_ROOT, "test")}

train_tokens_orig = {"ESD": os.path.join(DATASET_ROOT, "train_esd.txt")}
val_tokens_orig = {"ESD": os.path.join(DATASET_ROOT, "val_esd.txt")}
test_tokens_orig = {"ESD": os.path.join(DATASET_ROOT, "test_esd.txt")}

f0_file = os.path.join(DATASET_ROOT, "f0.pickle")
ease_embeddings_dir = os.path.join(DATASET_ROOT, "EASE_embeddings")

hparams = {
    "epochs": 200,
    "seed": 1234,
    "n_mel_channels": 80,
    "output_classes": 5,
    "encoder_embedding_dim": 128,
    "speaker_embedding_dim": 192,
    "emotion_embedding_dim": 128,
    "feed_back_last": True,
    "n_frames_per_step_decoder": 1,
    "decoder_rnn_dim": 512,
    "prenet_dim": [256, 256],
    "max_decoder_steps": 2000,
    "stop_threshold": 0.5,
    "attention_rnn_dim": 512,
    "attention_dim": 128,
    "attention_location_n_filters": 32,
    "attention_location_kernel_size": 17,
    "postnet_n_convolutions": 5,
    "postnet_dim": 512,
    "postnet_kernel_size": 5,
    "postnet_dropout": 0.1,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "grad_clip_thresh": 5.0,
    "batch_size": 32,
    "warmup": 7,
    "decay_rate": 0.5,
    "decay_every": 7
}

SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [9]:
# ----------------------------------------------------
# 2. DATASETS & COLLATOR DEFINITIONS
# ----------------------------------------------------
class MyDataset(Dataset):
    def __init__(self, folder, token_file):
        self.folder = folder
        
        # Recursive scan to find all wav files in all subfolders
        import glob
        wav_files = glob.glob(os.path.join(folder, "**", "*.wav"), recursive=True)
        # Keep only the basenames of the files
        self.wav_files = [os.path.basename(x) for x in wav_files]
        self.wav_files = sorted(list(set(self.wav_files))) # Remove any duplicates and sort
        
        self.sr = 16000
        self.tokens = {}
        
        with open(f0_file, 'rb') as handle:
            self.f0_feat = pickle.load(handle)
            
        with open(token_file) as f:
            lines = f.readlines()
            for l in lines:
                d = ast.literal_eval(l)
                name, tokens = d["audio"], d["hubert"]
                tokens_l = tokens.split(" ")
                filename = os.path.basename(name.replace("\\", "/"))
                self.tokens[filename] = np.array(tokens_l).astype(int)

    def __len__(self):
        return len(self.wav_files) 

    def getemolabel(self, file_name):
        file_name_val = int(file_name[5:-4])
        if file_name_val <= 350:
            return 0
        elif file_name_val > 350 and file_name_val <= 700:
            return 1
        elif file_name_val > 700 and file_name_val <= 1050:
            return 2
        elif file_name_val > 1050 and file_name_val <= 1400:
            return 3
        else:
            return 4

    def getspkrlabel(self, file_name):
        spkr_name = file_name[:4]
        speaker_dict = {}
        for ind in range(11, 21):
            speaker_dict["00"+str(ind)] = ind-11
            
        npy_filename = file_name.replace(".wav", ".npy")
        speaker_feature = np.load(os.path.join(ease_embeddings_dir, npy_filename))

        return speaker_feature, speaker_dict[spkr_name]
        
    def __getitem__(self, audio_ind): 
        wav_name = self.wav_files[audio_ind]
        class_id = self.getemolabel(wav_name)  
        
        # Find the absolute path since we are scanning recursively
        import glob
        matching_paths = glob.glob(os.path.join(self.folder, "**", wav_name), recursive=True)
        if len(matching_paths) > 0:
            audio_path = matching_paths[0]
        else:
            audio_path = os.path.join(self.folder, wav_name)
        
        (sig, sr) = torchaudio.load(audio_path)
        sig = sig.numpy()[0, :]
        
        tokens = self.tokens[wav_name]
        speaker_feat, speaker_label = self.getspkrlabel(wav_name)
        
        final_sig = sig
        f0 = self.f0_feat[wav_name]

        return final_sig, f0, tokens, class_id, speaker_feat, speaker_label, wav_name

def custom_collate(data):
    new_data = {"audio":[], "mask":[], "labels":[], "hubert":[], "f0":[], "speaker":[], "speaker_label":[], "names":[]}
    max_len_f0, max_len_hubert, max_len_aud = 0, 0, 0
    for ind in range(len(data)):
        max_len_aud = max(data[ind][0].shape[-1], max_len_aud)
        max_len_f0 = max(data[ind][1].shape[-1], max_len_f0)
        max_len_hubert = max(data[ind][2].shape[-1], max_len_hubert)
    for i in range(len(data)):
        final_sig = np.concatenate((data[i][0], np.zeros((max_len_aud-data[i][0].shape[-1]))), -1)
        f0_feat = np.concatenate((data[i][1], np.zeros((max_len_f0-data[i][1].shape[-1]))), -1)
        mask = data[i][2].shape[-1]
        hubert_feat = np.concatenate((data[i][2], 100*np.ones((max_len_f0-data[i][2].shape[-1]))), -1)
        labels = data[i][3]
        speaker_feat = data[i][4]
        speaker_label = data[i][5]
        names = data[i][6]
        new_data["audio"].append(final_sig)
        new_data["f0"].append(f0_feat)
        new_data["mask"].append(torch.tensor(mask))
        new_data["hubert"].append(hubert_feat)
        new_data["labels"].append(labels)
        new_data["speaker"].append(speaker_feat)
        new_data["speaker_label"].append(speaker_label)
        new_data["names"].append(names)

    return new_data

In [10]:
# ----------------------------------------------------
# 3. NETWORK DEFINITIONS (Robust to Batch Size = 1)
# ----------------------------------------------------
class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class WAV2VECModel(nn.Module):
    def __init__(self, wav2vec, output_dim, hidden_dim_emo):
        super().__init__()
        self.wav2vec = wav2vec
        embedding_dim = wav2vec.config.to_dict()['hidden_size']
        self.out = nn.Linear(hidden_dim_emo, output_dim)
        self.out_spkr = nn.Linear(hidden_dim_emo, 10)
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(in_channels=hidden_dim_emo, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv4 = nn.Conv1d(in_channels=hidden_dim_emo, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        
    def forward(self, aud, alpha):
        # Ensure aud is 2D: [batch_size, sequence_length]
        if aud.dim() == 1:
            aud = aud.unsqueeze(0)
        elif aud.dim() == 3:
            if aud.shape[0] == 1:
                aud = aud.squeeze(0)
            elif aud.shape[1] == 1:
                aud = aud.squeeze(1)

        hidden_all = list(self.wav2vec(aud).hidden_states)
        embedded = sum(hidden_all)
        embedded = embedded.permute(0, 2, 1)

        emo_embedded = self.relu(self.conv1(embedded))
        emo_embedded = self.relu(self.conv2(emo_embedded))
        emo_embedded = emo_embedded.permute(0, 2, 1)
        emo_hidden = torch.mean(emo_embedded, 1).squeeze(1)

        out_emo = self.out(emo_hidden)

        reverse_feature = ReverseLayerF.apply(embedded, alpha)
        embedded_spkr = self.relu(self.conv3(reverse_feature))
        embedded_spkr = self.relu(self.conv4(embedded_spkr))
        hidden_spkr = torch.mean(embedded_spkr, -1).squeeze(-1)
        output_spkr = self.out_spkr(hidden_spkr)
        
        return out_emo, output_spkr, emo_hidden, emo_embedded

class CrossAttentionModel(nn.Module):
    def __init__(self, hidden_dim_q, hidden_dim_k):
        super().__init__()
        HIDDEN_SIZE = 256
        NUM_ATTENTION_HEADS = 4
        self.inter_dim = HIDDEN_SIZE//NUM_ATTENTION_HEADS
        self.num_heads = NUM_ATTENTION_HEADS
        self.fc_q = nn.Linear(hidden_dim_q, self.inter_dim*self.num_heads)
        self.fc_k = nn.Linear(hidden_dim_k, self.inter_dim*self.num_heads)
        self.fc_v = nn.Linear(hidden_dim_k, self.inter_dim*self.num_heads)

        self.multihead_attn = nn.MultiheadAttention(self.inter_dim*self.num_heads,
                                                    self.num_heads,
                                                    dropout = 0.5,
                                                    bias = True,
                                                    batch_first=True)
                                                                                                           
        self.dropout = nn.Dropout(0.5)
        self.layer_norm = nn.LayerNorm(hidden_dim_q, eps = 1e-6)
        self.layer_norm_1 = nn.LayerNorm(hidden_dim_q, eps = 1e-6)
        self.fc = nn.Linear(self.inter_dim*self.num_heads, hidden_dim_q)
        self.fc_1 = nn.Linear(hidden_dim_q, hidden_dim_q)
        self.relu = nn.ReLU()
    
    def forward(self, query_i, key_i, value_i):
        query = self.fc_q(query_i)
        key = self.fc_k(key_i)
        value = self.fc_v(value_i)
        cross, _ = self.multihead_attn(query, key, value, need_weights = True)
        skip = self.fc(cross)
 
        skip += query_i
        skip = self.relu(skip)
        skip = self.layer_norm(skip)

        new = self.fc_1(skip)
        new += skip
        new = self.relu(new)
        out = self.layer_norm_1(new)
        return out

class PitchModel(nn.Module):
    def __init__(self, hparams):
        super(PitchModel, self).__init__()
        self.processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-robust-ft-swbd-300h")
        self.wav2vec = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-robust-ft-swbd-300h", output_hidden_states=True).to(device)
        self.encoder = WAV2VECModel(self.wav2vec, 5, hparams["emotion_embedding_dim"]).to(device)
        self.embedding = nn.Embedding(101, 128, padding_idx=100).to(device)        
        self.fusion = CrossAttentionModel(128, 128).to(device)
        self.linear_layer = nn.Linear(128, 1).to(device)
        self.leaky = nn.LeakyReLU().to(device)
        self.cnn_reg1 = nn.Conv1d(128, 128, kernel_size=(3,), padding=1).to(device)
        self.cnn_reg2 = nn.Conv1d(128, 1, kernel_size=(1,), padding=0).to(device)
        self.speaker_linear = nn.Linear(128, 128).to(device)

    def forward(self, aud, tokens=None, speaker=None, lengths=None, alpha=1.0):
        # Allow running with tokens=None for emo_feats extraction
        if tokens is not None:
            hidden = self.embedding(tokens.int())
        inputs = self.processor(aud, sampling_rate=16000, return_tensors="pt")
        
        # Squeeze inputs['input_values'] to always be 2D [batch_size, sequence_length]
        input_values = inputs['input_values']
        if input_values.dim() == 3:
            if input_values.shape[0] == 1:
                input_values = input_values.squeeze(0)
            elif input_values.shape[1] == 1:
                input_values = input_values.squeeze(1)
        elif input_values.dim() == 1:
            input_values = input_values.unsqueeze(0)

        emo_out, spkr_out, emo_hidden, emo_embedded = self.encoder(input_values.to(device), alpha)
        
        if tokens is None:
            return emo_hidden

        speaker_temp = speaker.unsqueeze(1).repeat(1, emo_embedded.shape[1], 1)
        speaker_temp = self.speaker_linear(speaker_temp)
        emo_embedded = emo_embedded + speaker_temp
        pred_pitch = self.fusion(hidden, emo_embedded, emo_embedded)
        pred_pitch = pred_pitch.permute(0, 2, 1)
        pred_pitch = self.cnn_reg2(self.leaky(self.cnn_reg1(pred_pitch)))
        pred_pitch = pred_pitch.squeeze(1)
        mask = torch.arange(hidden.shape[1]).expand(hidden.shape[0], hidden.shape[1]).to(device) < lengths.unsqueeze(1)
        pred_pitch = pred_pitch.masked_fill(~mask, 0.0)
        mask = mask.int()

        return pred_pitch, emo_out, spkr_out, mask

In [11]:
# ----------------------------------------------------
# 4. RUN EXTRACTIONS
# ----------------------------------------------------
def get_features_and_contours():
    os.makedirs("/kaggle/working/f0_contours", exist_ok=True)
    os.makedirs("/kaggle/working/wav2vec_feats", exist_ok=True)
    os.makedirs("/kaggle/working/pred_DSDT_f0", exist_ok=True)

    print("Initializing model...")
    model = PitchModel(hparams)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.to(device)
    model.eval()

    test_dataset = MyDataset(test_datasets["ESD"], test_tokens_orig["ESD"])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate, num_workers=4)

    train_dataset = MyDataset(train_datasets["ESD"], train_tokens_orig["ESD"])
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate, num_workers=4)

    val_dataset = MyDataset(val_datasets["ESD"], val_tokens_orig["ESD"])
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate, num_workers=4)

    # test set
    print("Extracting Test Set features...")
    with torch.no_grad():
        for data in tqdm(test_loader):
            inputs = torch.tensor(data["audio"]).float().to(device)
            mask = torch.tensor(data["mask"]).to(device)
            tokens = torch.tensor(data["hubert"]).to(device)
            speaker = torch.tensor(data["speaker"]).float().to(device)
            names = data["names"]
            
            pitch_pred, _, _, _ = model(inputs, tokens, speaker, mask)
            pitch_pred = torch.exp(pitch_pred) - 1
            emo_hidden = model(inputs)
            
            for ind in range(len(names)):
                filename = os.path.basename(names[ind].replace("\\", "/")).replace(".wav", ".npy")
                np.save(os.path.join("/kaggle/working/f0_contours", filename), pitch_pred[ind, :mask[ind]].cpu().numpy())
                np.save(os.path.join("/kaggle/working/wav2vec_feats", filename), emo_hidden[ind, :].cpu().numpy())

    # train set
    print("Extracting Train Set features...")
    with torch.no_grad():
        for data in tqdm(train_loader):
            inputs = torch.tensor(data["audio"]).float().to(device)
            mask = torch.tensor(data["mask"]).to(device)
            tokens = torch.tensor(data["hubert"]).to(device)
            speaker = torch.tensor(data["speaker"]).float().to(device)
            names = data["names"]
            
            pitch_pred, _, _, _ = model(inputs, tokens, speaker, mask)
            pitch_pred = torch.exp(pitch_pred) - 1
            emo_hidden = model(inputs)
            
            for ind in range(len(names)):
                filename = os.path.basename(names[ind].replace("\\", "/")).replace(".wav", ".npy")
                np.save(os.path.join("/kaggle/working/f0_contours", filename), pitch_pred[ind, :mask[ind]].cpu().numpy())
                np.save(os.path.join("/kaggle/working/wav2vec_feats", filename), emo_hidden[ind, :].cpu().numpy())

    # val set
    print("Extracting Val Set features...")
    with torch.no_grad():
        for data in tqdm(val_loader):
            inputs = torch.tensor(data["audio"]).float().to(device)
            mask = torch.tensor(data["mask"]).to(device)
            tokens = torch.tensor(data["hubert"]).to(device)
            speaker = torch.tensor(data["speaker"]).float().to(device)
            names = data["names"]
            
            pitch_pred, _, _, _ = model(inputs, tokens, speaker, mask)
            pitch_pred = torch.exp(pitch_pred) - 1
            emo_hidden = model(inputs)
            
            for ind in range(len(names)):
                filename = os.path.basename(names[ind].replace("\\", "/")).replace(".wav", ".npy")
                np.save(os.path.join("/kaggle/working/f0_contours", filename), pitch_pred[ind, :mask[ind]].cpu().numpy())
                np.save(os.path.join("/kaggle/working/wav2vec_feats", filename), emo_hidden[ind, :].cpu().numpy())

    # F0 Convert phase
    print("Converting test F0 contours...")
    sources = ["0011_000021.wav", "0012_000022.wav", "0013_000025.wav",
               "0014_000032.wav", "0015_000034.wav", "0016_000035.wav",
               "0017_000038.wav", "0018_000043.wav", "0019_000023.wav",
               "0020_000047.wav"]

    test_loader_bs1 = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=custom_collate)

    with torch.no_grad():
        for source in sources:
            target_speaker = None
            # Find the speaker index for the target audio source
            for d in test_loader_bs1:
                if d["names"][0] == source:
                    target_speaker = torch.tensor(d["speaker"]).float().to(device)
                    break
            
            if target_speaker is None:
                continue

            for data in test_loader_bs1:
                inputs = torch.tensor(data["audio"]).float().to(device)
                mask = torch.tensor(data["mask"]).to(device)
                tokens = torch.tensor(data["hubert"]).to(device)
                names = data["names"]
                
                pitch_pred, _, _, _ = model(inputs, tokens, target_speaker, mask)
                pitch_pred = torch.exp(pitch_pred) - 1
                
                for ind in range(len(names)):
                    target_file_name = names[ind] + source.replace(".wav", ".npy")
                    target_file_name = os.path.basename(target_file_name.replace("\\", "/"))
                    np.save(os.path.join("/kaggle/working/pred_DSDT_f0", target_file_name), pitch_pred[ind, :mask[ind]].cpu().numpy())

    print("Archiving outputs...")
    os.system("zip -q -r /kaggle/working/zest_extracted_outputs.zip /kaggle/working/f0_contours /kaggle/working/wav2vec_feats /kaggle/working/pred_DSDT_f0")
    print("Done! You can now download /kaggle/working/zest_extracted_outputs.zip")

In [12]:
get_features_and_contours()

Initializing model...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

/tmp/ipykernel_58/926937352.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  self.f0_feat = pickle.load(handle)


Extracting Test Set features...


100%|██████████| 47/47 [03:44<00:00,  4.77s/it]


Extracting Train Set features...


100%|██████████| 469/469 [32:56<00:00,  4.21s/it]


Extracting Val Set features...


100%|██████████| 32/32 [02:31<00:00,  4.73s/it]


Converting test F0 contours...
Archiving outputs...
Done! You can now download /kaggle/working/zest_extracted_outputs.zip
